# Trace-Bench Smoke Runner (Stub + Real)

This notebook validates Trace-Bench in two modes:

- **StubLLM**: deterministic, no API keys
- **Real LLM**: requires a user-provided API key (via Colab Secrets)

It also shows the standardized run artifacts produced by the CLI.

## Expected Outputs (Quick Verification)

You should see the following signals if the notebook is working correctly:

- **Stub smoke run** completes with a new `runs/<run_id>/` folder.
- `config.snapshot.yaml`, `env.json`, `results.csv`, `events.jsonl` exist in that folder.
- `results.csv` shows at least one row with `task=example:greeting_stub` and `status=trained`.
- **Real-LLM smoke** completes (if API key is set) and `results.csv` shows `status=trained`.
- `pytest -q` ends with `passed` (LLM4AD optimizer tests run only when `OPENAI_API_KEY` is set).

In [8]:
# Mount Drive (optional) + compute persistent runs_dir
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


def bench_dir(project="bench", sub="trace_bench", local="/content/bench"):
    drive = Path("/content/drive/MyDrive")
    root = drive if drive.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runs dir: /content/drive/MyDrive/bench/2026-02-09/trace_bench


In [9]:
# Clone repos side-by-side (Trace-Bench + OpenTrace)
!git clone --depth 1 --branch runner-foundation https://github.com/guru-code-expert/Trace-Bench.git
!git clone --depth 1 --branch experimental https://github.com/guru-code-expert/OpenTrace.git

%cd Trace-Bench

# System + Python deps
!apt-get update -y && apt-get install -y graphviz
!python -m pip install -U pip
!python -m pip install pyyaml pytest numpy matplotlib graphviz litellm==1.75.0

Cloning into 'Trace-Bench'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (302/302), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 302 (delta 37), reused 270 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (302/302), 3.83 MiB | 22.68 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Cloning into 'OpenTrace'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 228 (delta 17), reused 114 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (228/228), 4.73 MiB | 10.09 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/Trace-Bench/Trace-Bench
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archiv

In [10]:
# Optional: list tasks (external bench discovery)
!python -m trace_bench list-tasks --root LLM4AD/benchmark_tasks | head -n 30

circle_packing
online_bin_packing_local
optimization/tsp_gls_2O
optimization/set_cover_construct
optimization/tsp_construct
optimization/bp_2d_construct
optimization/online_bin_packing_2O
optimization/cflp_construct
optimization/vrptw_construct
optimization/online_bin_packing
optimization/knapsack_construct
optimization/pymoo_moead
optimization/cvrp_construct
optimization/jssp_construct
optimization/bp_1d_construct
optimization/admissible_set
optimization/qap_construct
optimization/ovrp_construct
optimization/co_bench/open_shop_scheduling_co_bench
optimization/co_bench/generalised_assignment_problem_co_bench
optimization/co_bench/flow_shop_scheduling_co_bench
optimization/co_bench/set_partitioning_co_bench
optimization/co_bench/maximal_independent_set_co_bench
optimization/co_bench/container_loading_co_bench
optimization/co_bench/equitable_partitioning_problem_co_bench
optimization/co_bench/p_median_uncapacitated_co_bench
optimization/co_bench/crew_scheduling_co_bench
optimization/co_b

In [11]:
%%bash
cd /content/Trace-Bench

# Stub smoke (internal example task for deterministic output)
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config configs/smoke.yaml --runs-dir "$RUNS_DIR"

In [12]:
# Inspect latest run artifacts
import glob, json, pathlib, pandas as pd

latest = sorted(glob.glob(f"{RUNS_DIR}/*"))[-1]
p = pathlib.Path(latest)
print(p)

print((p / "config.snapshot.yaml").read_text()[:400])
print(json.loads((p / "env.json").read_text()).keys())

pd.read_csv(p / "results.csv").head()

/content/drive/MyDrive/bench/2026-02-09/trace_bench/6ce874e0-aff9-497d-8781-a5f8d8bde1c1
run_id: 6ce874e0-aff9-497d-8781-a5f8d8bde1c1
runs_dir: /content/drive/MyDrive/bench/2026-02-09/trace_bench
mode: stub
seed: 123
tasks:
- example:greeting_stub
trainers:
- PrioritySearch
eval_kwargs: {}
trainer_kwargs: {}

dict_keys(['captured_at', 'env', 'runtime', 'git'])


,timestamp,task,trainer,status,score,feedback
0,2026-02-09T10:30:50.025240Z,example:greeting_stub,PrioritySearch,train_error,1.0,Correct


In [13]:
%%bash
cd /content/Trace-Bench

# Optional: external LLM4AD smoke (may yield low score if template fails)
cat > configs/smoke_llm4ad.yaml <<'YAML'
runs_dir: runs
mode: stub
seed: 123
tasks:
  - circle_packing
trainers:
  - PrioritySearch
YAML

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config configs/smoke_llm4ad.yaml --runs-dir "$RUNS_DIR"

## Real LLM (requires API key)

Add `OPENAI_API_KEY` in **Colab Secrets** and run the cells below.

In [14]:
# Load API key from Colab Secrets
from google.colab import userdata
import os

key = userdata.get("OPENAI_API_KEY")
if not key:
    raise RuntimeError("Missing OPENAI_API_KEY secret in Colab")

os.environ["OPENAI_API_KEY"] = key
os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
os.environ["TRACE_LITELLM_MODEL"] = "gpt-4o-mini"

In [15]:
%%bash
cd /content/Trace-Bench

# Real-LLM smoke (internal example task)
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config configs/smoke_real.yaml --runs-dir "$RUNS_DIR"

In [16]:
%%bash
cd /content/Trace-Bench

# Pytest (LLM4AD optimizer test runs only if OPENAI_API_KEY is set)
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m pytest -q

.........................                                                [100%]
=============================== warnings summary ===============================
tests/test_lite_optimize_llm4ad.py::test_lite_optimize_llm4ad_task[task0]
tests/test_lite_optimize_llm4ad.py::test_lite_optimize_llm4ad_task[task1]
  /usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
    PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{\n"reas...: None}, annotations=[]), input_type=Message])
    PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
    return self.__pydantic_serializer__.to_python(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.htm